In [193]:
import nbiatoolkit
print(nbiatoolkit.__version__)



0.29.2


In [194]:

import sys
import os
import numpy as np
import pandas as pd
sys.path.append(os.path.curdir + '/..')
from Dataset.custom_dataset import get_data_loaders
from pycox.evaluation import EvalSurv
from statsmodels.stats.proportion import proportion_confint
import Dataset
print(Dataset.__name__)



Dataset


In [195]:
# def sample_from_dist(params, distributions):
#     """
#     Generates samples from specified distribution parameters.
    
#     Args:
#         params (torch.Tensor): Tensor of distribution parameters, expected shape [batch_size, 2]
#                                where params[:, 0] = alpha and params[:, 1] = lambda for Weibull.
#         distributions (list): List specifying distribution types, e.g., ['weibull'].

#     Returns:
#         numpy.ndarray: Array of generated samples from specified distributions.
#     """
#     samples = []
#     for dist in distributions:
#         if dist == 'weibull':
#             alpha, lam = params[:, 0], params[:, 1]  # Extract Weibull parameters
#             # Sample from Weibull distribution
#             weibull_samples = lam * (-torch.log(torch.rand_like(alpha))) ** (1 / alpha)
#             samples.append(weibull_samples.cpu().numpy())  # Convert samples to numpy
#         else:
#             raise NotImplementedError(f"Unknown distribution type: {dist}")
    
#     # Concatenate all samples along the batch dimension
#     return np.concatenate(samples, axis=0)

In [196]:

import torch
import torch.nn as nn
import torch.nn.functional as F

class Encoder3D(nn.Module):
    def __init__(self, in_channels, base_channels):
        super(Encoder3D, self).__init__()
        
        self.encoder = nn.Sequential(
            nn.Conv3d(in_channels, base_channels, kernel_size=3, padding=1),
            nn.BatchNorm3d(base_channels),
            nn.ReLU(inplace=True),
            nn.Conv3d(base_channels, base_channels, kernel_size=3, padding=1),
            nn.BatchNorm3d(base_channels),
            nn.ReLU(inplace=True),
            nn.MaxPool3d(kernel_size=2)  # Downsample by 2
        )
    
    def forward(self, x):
        return self.encoder(x)
class Decoder3D(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(Decoder3D, self).__init__()
        
        self.decoder = nn.Sequential(
            nn.ConvTranspose3d(in_channels, out_channels, kernel_size=2, stride=2),  # Upsample by 2
            nn.Conv3d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm3d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv3d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm3d(out_channels),
            nn.ReLU(inplace=True),
        )
    
    def forward(self, x):
        return self.decoder(x)

class TimeDecoder(nn.Module):
    def __init__(self, input_dim):
        super(TimeDecoder, self).__init__()
        # Intermediate dimension
        intermediate_dim = input_dim * 2  # You can adjust this multiplier as necessary
        
        # Define a series of Linear layers and ReLUs
        self.layers = nn.Sequential(
            nn.Linear(input_dim, intermediate_dim),
            nn.ReLU(),
            nn.Linear(intermediate_dim, intermediate_dim // 2),
            nn.ReLU(),
            nn.Linear(intermediate_dim // 2, 2)  # Outputs the shape and scale parameters
        )
        self.softplus = nn.Softplus() 

    def forward(self, z):
        return self.softplus(self.layers(z))



In [197]:

class LatentSpaceGaussian(nn.Module):
    def __init__(self, latent_dim):
        super(LatentSpaceGaussian, self).__init__()
        self.latent_dim = latent_dim
        self.latent_params = 2 * latent_dim  # Two parameters are needed for each Gaussian distribution


    def get_latent_params(self, x):
        x = x.view(-1, 2, self.latent_dim)
        mu = x[:, 0, :]
        log_var = x[:, 1, :]
        return mu, log_var


    def sample_latent(self, latent_params):
        mu, log_var = latent_params
        std = torch.exp(0.5 * log_var)
        eps = torch.randn_like(std)  # `randn_like` as we need the same size
        z = mu + (eps * std)  # sampling as if coming from the input space
        return z


    def kl_loss(self, latent_params):
        mu, log_var = latent_params
        kl = -0.5 * torch.sum(1 + log_var - mu.pow(2) - log_var.exp())  # Kullback-Leibler divergence
        return kl / mu.shape[0]


In [198]:

class LogLikelihoodLossWithCensoring(nn.Module):
    def __init__(self, dist_type):
        """
        Initialize the log-likelihood loss with censoring class.
        
        Args:
            dist_type (str): Distribution type to use for survival prediction (currently only supports 'weibull').
        """
        super(LogLikelihoodLossWithCensoring, self).__init__()
        self.dist_type = dist_type

    def forward(self, inputs, survival_times, risk_status):
        """
        Compute the log-likelihood loss with censoring.
        
        Args:
            inputs (Tensor): Weibull parameters (alpha and lambda) predicted by the model, shape [batch_size, 2].
            survival_times (Tensor): Actual survival times for each sample, shape [batch_size].
            risk_status (Tensor): Binary censoring indicators (1 if event occurred, 0 if censored), shape [batch_size].

        Returns:
            loss (Tensor): Mean log-likelihood loss with censoring.
        """
        survival_times = torch.squeeze(survival_times)
        risk_status = torch.squeeze(risk_status)

        # Extract Weibull parameters from inputs
        alpha = inputs[:, 0]
        lam = inputs[:, 1]

        # Calculate the survival log-probabilities
        survival_term = -torch.pow(survival_times / lam, alpha)
        
        # Calculate the hazard function where risk_status is 1
        hazard_term = torch.log(alpha / lam) + (alpha - 1) * torch.log(survival_times / lam)

        # Apply the censoring: use hazard term only where risk status is 1 (event occurred)
        log_likelihood_loss = -torch.sum(hazard_term[risk_status > 0.5]) - torch.sum(survival_term)
        print(log_likelihood_loss)
        # Compute the mean loss across the batch
        loss = log_likelihood_loss / inputs.size(0)
        
        # Ensure finite loss and handle NaNs/INFs
        if not torch.isfinite(loss):
            raise ValueError("Non-finite loss encountered, check inputs for stability issues.")

        return loss



In [199]:
# # Create dummy variables
# batch_size = 5  # Example batch size
# inputs = torch.tensor([[2.0, 1.0],  # alpha, lambda for sample 1
#                        [1.5, 0.5],  # alpha, lambda for sample 2
#                        [3.0, 2.0],  # alpha, lambda for sample 3
#                        [1.0, 1.0],  # alpha, lambda for sample 4
#                        [2.5, 1.5]], # alpha, lambda for sample 5
#                       dtype=torch.float32)  # Weibull parameters (alpha, lambda)

# survival_times = torch.tensor([10.0, 5.0, 15.0, 20.0, 25.0], dtype=torch.float32)  # Survival times
# risk_status = torch.tensor([1, 1, 0, 1, 0], dtype=torch.float32)  # Risk status (1 if event occurred, 0 if censored)

# # Create an instance of the loss class
# loss_fn = LogLikelihoodLossWithCensoring(dist_type='weibull')

# # Compute the loss
# loss = loss_fn(inputs, survival_times, risk_status)

# # Print the loss
# print(f"Log-likelihood loss: {loss.item()}")

In [200]:
class UNet3D(nn.Module):
    def __init__(self, in_channels, out_channels, base_channels, latent_dim=None):
        super(UNet3D, self).__init__()
        
        self.latent_dim = latent_dim
        self.rec_loss = nn.MSELoss() 
        self.time_loss = LogLikelihoodLossWithCensoring(dist_type='weibull')
        # Encoder and decoder layers
        self.enc1 = Encoder3D(in_channels, base_channels)
        self.enc2 = Encoder3D(base_channels, base_channels * 2)
        self.enc3 = Encoder3D(base_channels * 2, base_channels * 4)
        self.enc4 = Encoder3D(base_channels * 4, base_channels * 8)
        
        if latent_dim is not None:
            self.latent_space = LatentSpaceGaussian(latent_dim)
            # Calculate size for linear layer to output latent parameters
            self.latent_mu = nn.Linear(base_channels * 8 * 8 * 8 * 4, 2 * latent_dim)  # Assuming the output from the encoder is [4, 128, 8, 8, 4]

            #self.fc_decode = nn.Linear(latent_dim, base_channels * 8 * 8 * 4)
            self.fc_decode = nn.Linear(latent_dim, 128 * 8 * 8 * 4)  # Outputs channels to match the decoder's expectations

        
        self.dec4 = Decoder3D(base_channels * 8, base_channels * 4)
        self.dec3 = Decoder3D(base_channels * 4, base_channels * 2)
        self.dec2 = Decoder3D(base_channels * 2, base_channels)
        self.dec1 = Decoder3D(base_channels, out_channels)

        if latent_dim is not None:
            self.time_decoder = TimeDecoder(latent_dim)
    
    def forward(self, x):
        batch_size, _, H, W, D = x.shape 
        #print("Input shape:", x.shape)  # Print input shape, expect (4, 1, 128, 128, 64)

        enc1 = self.enc1(x)
        #print("After enc1 shape:", enc1.shape)  # Expect (4, base_channels, 64, 64, 32) after downsampling

        enc2 = self.enc2(enc1)
        #print("After enc2 shape:", enc2.shape)  # Expect (4, base_channels*2, 32, 32, 16)

        enc3 = self.enc3(enc2)
        #print("After enc3 shape:", enc3.shape)  # Expect (4, base_channels*4, 16, 16, 8)

        enc4 = self.enc4(enc3)
        #print("After enc4 shape:", enc4.shape)  # Expect (4, base_channels*8, 8, 8, 4)
        
        out_data = {}  # Dictionary to store outputs
        
        if self.latent_dim is not None:
            # Flatten and pass through latent layers
            enc4_flat = enc4.view(enc4.size(0), -1)
            #print("enc4_flat shape:", enc4_flat.shape)  # Expect (4, base_channels*8*8*8)

            # Get the latent parameters directly from the linear layer
            latent_params = self.latent_mu(enc4_flat)
            #print("latent_params shape:", latent_params.shape)  # Expect (4, 2 * latent_dim)

            mu, log_var = self.latent_space.get_latent_params(latent_params)
            #print("mu shape:", mu.shape)  # Expect (4, latent_dim)
            #print("log_var shape:", log_var.shape)  # Expect (4, latent_dim)
            
            z = self.latent_space.sample_latent((mu, log_var))
            #print("z shape:", z.shape)  # Expect (4, latent_dim)
            
            # Decode the latent sample
            enc4_decoded = self.fc_decode(z)
            #print("enc4_decoded shape:", enc4_decoded.shape)  # Expect (4, base_channels * 8 * 8 * 8)

            enc4 = enc4_decoded.view(enc4_decoded.size(0), -1, 8, 8, 4)
            #print("enc4 reshaped shape:", enc4.shape)  # Expect (4, base_channels * 8, 8, 8, 4)
            
            # Predict Weibull parameters
            weibull_params = self.time_decoder(z)
            #print("weibull_params shape:", weibull_params.shape)  # Expect (4, 2) for Weibull params
            
            # Add to out_data dictionary
            out_data['latent_sample'] = z
            out_data['latent_params'] = latent_params
            out_data['time_params'] = weibull_params
            # out_data['mu'] = mu
            # out_data['log_var'] = log_var
        
        # Decoding path
        dec4 = self.dec4(enc4)
        #print("dec4 shape:", dec4.shape)  # Expect (4, base_channels*4, 16, 16, 8)

        dec3 = self.dec3(dec4 + enc3)
        #print("dec3 shape:", dec3.shape)  # Expect (4, base_channels*2, 32, 32, 16)

        dec2 = self.dec2(dec3 + enc2)
        #print("dec2 shape:", dec2.shape)  # Expect (4, base_channels, 64, 64, 32)

        dec1 = self.dec1(dec2 + enc1)
        #print("dec1 shape:", dec1.shape)  # Expect (4, out_channels, 128, 128, 64)
        
        # Reconstruction output
        out_data['reconstruction'] = dec1
        
        return out_data

    @staticmethod
    def sample_from_dist(params, distributions):
        """
        Generates samples from specified distribution parameters.
        
        Args:
            params (torch.Tensor): Tensor of distribution parameters, expected shape [batch_size, 2]
                                where params[:, 0] = alpha and params[:, 1] = lambda for Weibull.
            distributions (list): List specifying distribution types, e.g., ['weibull'].

        Returns:
            numpy.ndarray: Array of generated samples from specified distributions.
        """
        samples = []
        for dist in distributions:
            if dist == 'weibull':
                alpha, lam = params[:, 0], params[:, 1]  # Extract Weibull parameters
                alpha = torch.clamp(alpha, min=0.001)  # Prevent alpha from being zero or negative
                lam = torch.clamp(lam, min=0.001)
                # Sample from Weibull distribution
                weibull_samples = lam * (-torch.log(torch.rand_like(alpha))) ** (1 / alpha)
                if (weibull_samples <= 0).any():
                    print("Non-positive samples detected:", weibull_samples[weibull_samples <= 0])

                samples.append(weibull_samples.detach().cpu().numpy())  # Convert samples to numpy
            else:
                raise NotImplementedError(f"Unknown distribution type: {dist}")
        
        # Concatenate all samples along the batch dimension
        return np.concatenate(samples, axis=0)

    def predict_time(self, x, device=torch.device('cpu')):
        """
        Predicts Weibull time distribution parameters and samples for survival analysis
        using only image data.
        
        Args:
            x (torch.Tensor): Input image tensor of shape [batch_size, channels, H, W, D].
            device (torch.device, optional): Device to run the computations on (e.g., 'cpu' or 'cuda').

        Returns:
            dict: Dictionary with keys 'z', 'time_params', and 'time_samples':
                - 'z': Latent representation from image input.
                - 'time_params': Predicted Weibull distribution parameters.
                - 'time_samples': Sampled time prediction based on Weibull distribution.
        """
        # Ensure input is on the correct device
        x = x.to(device)
        
        # Forward pass through the network to get latent representation and time parameters
        out_data = self.forward(x)
        
        # Get latent variable z from the output data dictionary
        z = out_data['latent_sample']  # Latent representation
        time_params = out_data['weibull_params']  # Weibull distribution parameters
        
        # Generate a survival time sample from the Weibull distribution
        time_samples = self.sample_from_dist(time_params, distributions=['weibull'])
        
        return {
            'z': z.detach().cpu().numpy(),  # Detach and convert to NumPy array
            'time_params': time_params.detach().cpu().numpy(),
            'time_samples': time_samples
        }


    def predict_just_time(self, x):
        """Predicts only the time samples based on the input covariates."""
        # Call the predict_time method to get time samples
        return self.predict_time(x)['time_samples']


    @staticmethod
    def bern_conf_interval(n, mean, ibs=False):
        # Confidence interval
        ci_bot, ci_top = proportion_confint(count=mean * n, nobs=n, alpha=0.1, method='beta')
        if mean < 0.5 and not ibs:
            ci_bot_2 = 1 - ci_top
            ci_top = 1 - ci_bot
            ci_bot = ci_bot_2
            mean = 1 - mean

        return np.round(ci_bot, 4), mean, np.round(ci_top, 4)

    @staticmethod
    def obtain_c_index(surv_f, time, censor):
        # Evaluate using PyCox c-index
        ev = EvalSurv(surv_f, time.flatten(), censor.flatten(), censor_surv='km')
        ci = ev.concordance_td()

        # Obtain also ibs
        time_grid = np.linspace(time.min(), time.max(), 100)
        ibs = ev.integrated_brier_score(time_grid)
        return ci, ibs
        
    def calculate_risk(self, time_train, x_val, censor_val):
        """Calculate risk CDF at specified time points, compute C-index and IBS with confidence intervals."""
        times = np.unique(time_train)
        out_data = self.predict_time(x_val)
        pred_risk = np.zeros((out_data['time_params'].shape[0], len(times)))

        for sample in range(pred_risk.shape[0]):
            # Directly assume the Weibull parameters are used
            alpha = out_data['time_params'][sample, 0]
            lam = out_data['time_params'][sample, 1]
            pred_risk[sample, :] = 1 - np.exp(-np.power(times / lam, alpha))

        # Calculate C-index and IBS along with confidence intervals
        ci, ibs = self.obtain_c_index(pd.DataFrame(1 - pred_risk.T, index=times), np.array(time_train), censor_val)
        ci_conf_intervals = self.bern_conf_interval(len(np.array(time_train)), ci)
        ibs_conf_intervals = self.bern_conf_interval(len(np.array(time_train)), ibs, ibs=True)

        return ci_conf_intervals, ibs_conf_intervals

    def fit_epoch(self, train_loader, test_loader, optimizer, device=None):
        # Automatically determine device if not provided
        if device is None:
            device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

        epoch_results = {
            'loss_tr': 0.0, 'loss_va': 0.0, 'kl_tr': 0.0, 'kl_va': 0.0,
            'll_cov_tr': 0.0, 'll_cov_va': 0.0, 'll_time_tr': 0.0, 'll_time_va': 0.0,
            'ci_va': 0.0, 'ibs_va': 0.0
        }

        # Train epoch
        self.train()
        for batch in train_loader:
            img_train_batch, time_train_batch, censor_train_batch = batch

            img_train_batch = img_train_batch.to(device)
            time_train_batch = time_train_batch.to(device)
            censor_train_batch = censor_train_batch.to(device)

            # Generate output params
            out = self.forward(img_train_batch)

            # Compute losses
            optimizer.zero_grad()
            loss_kl = self.latent_space.kl_loss(out['latent_params'])
            loss_img = self.rec_loss(out['reconstruction'], img_train_batch)  # Assuming mask is not provided
            loss_time = self.time_loss(out['time_params'], time_train_batch, censor_train_batch)
            loss = loss_kl + loss_img + loss_time
            loss.backward()
            optimizer.step()

            # Save training data
            epoch_results['loss_tr'] += loss.item()
            epoch_results['kl_tr'] += loss_kl.item()
            epoch_results['ll_cov_tr'] += loss_img.item()  # updated to use img
            epoch_results['ll_time_tr'] += loss_time.item()

        # Validation step
        self.eval()
        with torch.no_grad():
            for batch in test_loader:
                img_val, time_val, censor_val = batch

                img_val = img_val.to(device)
                time_val = time_val.to(device)
                censor_val = censor_val.to(device)

                out = self.forward(img_val)
                loss_kl = self.latent_space.kl_loss(out['latent_params'])
                loss_img = self.rec_loss(out['reconstruction'], img_train_batch) # Assuming mask is not provided
                loss_time = self.time_loss(out['time_params'], time_val, censor_val)
                loss = loss_kl + loss_img + loss_time

                # Save validation data
                epoch_results['loss_va'] += loss.item()
                epoch_results['kl_va'] += loss_kl.item()
                epoch_results['ll_cov_va'] += loss_img.item()  # updated to use img
                epoch_results['ll_time_va'] += loss_time.item()

        return epoch_results



    def fit(self, train_loader, test_loader, n_epochs=50, lr=0.001, batch_size=3, device=None):
        if device is None:
            device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        # Define optimizer
        optimizer = torch.optim.Adam(self.parameters(), lr=lr)

        # Track training stats
        training_stats = {
            'loss_tr': [], 'loss_va': [], 'kl_tr': [], 'kl_va': [], 
            'll_img_tr': [], 'll_img_va': [], 'll_time_tr': [], 'll_time_va': [],
            'ci_va': [], 'ibs_va': []
        }
        best_loss = np.inf
        best_model = None
        best_ci, best_ibs = 0.0, 1.0  # Initialize best values for metrics

        for epoch in range(n_epochs):
            # Optionally adjust weights for KL and image reconstruction terms
            self.kl_w = 0.001  # Constant value for KL weight
            self.cov_w = 0.001  # Constant value for covariance weight

            # Training epoch
            epoch_results = self.fit_epoch(train_loader,test_loader, optimizer=optimizer, device=device)

            # Validation step
            self.eval()
            with torch.no_grad():
                total_loss_va, total_kl_va, total_img_va, total_time_va = 0, 0, 0, 0
                for img_val, survival_time_val, event_val in test_loader:
                    img_val = img_val.to(device)
                    survival_time_val = survival_time_val.to(device)
                    event_val = event_val.to(device)

                    out_val = self.forward(img_val)
                    loss_kl = self.latent_space.kl_loss(out_val['latent_params'])
                    loss_img = self.rec_loss(out_val['reconstruction'], img_val)  # Updated loss_img calculation
                    loss_time = self.time_loss(out_val['time_params'], survival_time_val, event_val)
                    loss_va = loss_kl + loss_img + loss_time

                    total_loss_va += loss_va.item()
                    total_kl_va += loss_kl.item()
                    total_img_va += loss_img.item()
                    total_time_va += loss_time.item()

                # Average validation losses
                n_val_batches = len(test_loader)
                epoch_results['loss_va'] = total_loss_va / n_val_batches
                epoch_results['kl_va'] = total_kl_va / n_val_batches
                epoch_results['ll_img_va'] = total_img_va / n_val_batches
                epoch_results['ll_time_va'] = total_time_va / n_val_batches

                # Calculate validation metrics
                epoch_results['ci_va'], epoch_results['ibs_va'] = self.calculate_risk(
                    [survival_time for _, survival_time, _ in test_loader],
                    [img for img, _, _ in test_loader],
                    [event for _, _, event in test_loader]
                )

                # Save stats
                for key in epoch_results:
                    training_stats[key].append(epoch_results[key])

                # Print progress at each epoch
                if epoch % 10 == 0:
                    print(f"Epoch {epoch} | Train Loss: {epoch_results['loss_tr']:.4f} | "
                          f"Val Loss: {epoch_results['loss_va']:.4f} | "
                          f"Val C-Index: {epoch_results['ci_va']:.4f}")

                # Save the best model based on validation loss
                if epoch_results['loss_va'] < best_loss:
                    best_loss = epoch_results['loss_va']
                    best_model = self.state_dict()
                    best_ci = epoch_results['ci_va']
                    best_ibs = epoch_results['ibs_va']

        # Load the best model before returning
        if best_model is not None:
            self.load_state_dict(best_model)
        training_stats['ci_va'].append(best_ci)
        training_stats['ibs_va'].append(best_ibs)

        return training_stats


In [201]:
import matplotlib.pyplot as plt

def plot_training_stats(training_stats, fig_path_prefix="training_stats", x_label="Epochs"):
    """
    Plot training and validation losses and metrics over epochs.
    
    Args:
        training_stats (dict): Dictionary with keys as metric names and values as lists of metric values per epoch.
        fig_path_prefix (str): Prefix for the path to save the figures.
        x_label (str): Label for the x-axis, usually 'Epochs'.
    """
    # Plot each type of loss in a separate figure
    for stat_key in ['loss_tr', 'loss_va', 'kl_tr', 'kl_va', 'll_img_tr', 'll_img_va', 'll_time_tr', 'll_time_va']:
        fig_path = f"{fig_path_prefix}_{stat_key}.png"
        title = f"{stat_key.replace('_', ' ').title()} Over Epochs"
        plot_model_losses(training_stats[stat_key], training_stats[stat_key.replace('tr', 'va')], fig_path, title, x_label)

    # Plot C-Index and IBS in the same figure
    plt.figure(figsize=(10, 6))
    plt.plot(training_stats['ci_va'], label='C-Index (Validation)', color='blue')
    plt.plot(training_stats['ibs_va'], label='IBS (Validation)', color='green')
    plt.xlabel(x_label)
    plt.ylabel("Metric Value")
    plt.title("C-Index and IBS Over Epochs")
    plt.legend(loc="upper right")
    plt.savefig(f"{fig_path_prefix}_metrics.png")
    plt.close()

def plot_model_losses(train_loss, val_loss, fig_path, title, x_label='Epochs'):
    """
    Plot train and validation loss for a given metric.
    
    Args:
        train_loss (list): List of training losses over epochs.
        val_loss (list): List of validation losses over epochs.
        fig_path (str): Path to save the figure.
        title (str): Title of the plot.
        x_label (str): Label for the x-axis, default is 'Epochs'.
    """
    plt.figure(figsize=(10, 6))
    plt.semilogy(train_loss, label='Train')
    plt.semilogy(val_loss, label='Validation')
    plt.title(title)
    plt.xlabel(x_label)
    plt.ylabel("Loss")
    plt.legend(loc='upper right')
    plt.savefig(fig_path)
    plt.close()


In [202]:
import torch

# Set device for training (GPU if available, otherwise CPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)
# Initialize model and move it to the specified device
model = UNet3D(in_channels=1, out_channels=1, base_channels=16, latent_dim=16).to(device)

# # Example usage
train_loader, test_loader = get_data_loaders(
    root_dir='/home/nikhil/g17p3/g17-p3/NBIA-Download',
    excel_path='/home/nikhil/g17p3/g17-p3/Dataset/crlm.xlsx',
    batch_size=2,
    test_size=0.25
)

# Now you can use train_loader and test_loader in your model
print("len(train_loader)",len(train_loader))
print("len(test_loader)",len(test_loader))


# Train the model using the `fit` method
training_stats = model.fit(train_loader, test_loader)

# Plot losses and metrics if needed (using the existing `plot_model_losses` or a similar plotting function)
plot_model_losses(training_stats['loss_tr'], training_stats['loss_va'], 'loss_plot.png', title="Training and Validation Losses")
plot_model_losses(training_stats['ci_va'], training_stats['ibs_va'], 'metrics_plot.png', title="C-Index and IBS")


cuda
len(train_loader) 74
len(test_loader) 25
NIfTI file already exists at /home/nikhil/g17p3/g17-p3/NBIA-Download/CRLM-CT-1156/nifti/output.nii.gz. Skipping conversion.


Unknown extension is not supported and will be removed


NIfTI file already exists at /home/nikhil/g17p3/g17-p3/NBIA-Download/CRLM-CT-1175/nifti/output.nii.gz. Skipping conversion.
metatensor(30.6603, device='cuda:0', dtype=torch.float64, grad_fn=<AliasBackward0>)
NIfTI file already exists at /home/nikhil/g17p3/g17-p3/NBIA-Download/CRLM-CT-1141/nifti/output.nii.gz. Skipping conversion.
NIfTI file already exists at /home/nikhil/g17p3/g17-p3/NBIA-Download/CRLM-CT-1193/nifti/output.nii.gz. Skipping conversion.
metatensor(nan, device='cuda:0', dtype=torch.float64, grad_fn=<AliasBackward0>)


ValueError: Non-finite loss encountered, check inputs for stability issues.

In [ ]:
Cell In[97], line 24
     23 def kl_loss(self, latent_params):
---> 24     mu, log_var = latent_params
     25     kl = -0.5 * torch.sum(1 + log_var - mu.pow(2) - log_var.exp())  # Kullback-Leibler divergence
     26     return kl / mu.shape[0]

ValueError: too many values to unpack (expected 2)

In [ ]:
--> 242 loss_time = self.time_loss(out['time_params'], time_train_batch, censor_train_batch)
    243 loss = loss_kl + loss_img + loss_time
    244 loss.backward()

File ~/g17p3/g17-p3/g15p3/lib/python3.11/site-packages/torch/nn/modules/module.py:1518, in Module._wrapped_call_impl(self, *args, **kwargs)
   1516     return self._compiled_call_impl(*args, **kwargs)  # type: ignore[misc]
...
     44 if not torch.isfinite(loss):
---> 45     raise ValueError("Non-finite loss encountered, check inputs for stability issues.")
     47 return loss

ValueError: Non-finite loss encountered, check inputs for stability issues.

In [ ]:
import torch

# Set parameters
in_channels = 1            # Number of input channels
out_channels = 1           # Number of output channels
base_channels = 16         # Base channels for the model
latent_dim = 8             # Latent dimension for the VAE

# Create an instance of the UNet3D model
model = UNet3D(in_channels, out_channels, base_channels, latent_dim)

# Create a random tensor with the shape [4, 1, 128, 128, 64]
input_tensor = torch.rand(2, 1, 128, 128, 64)

# Pass the random input through the model
output_data = model(input_tensor)



Input shape: torch.Size([2, 1, 128, 128, 64])
After enc1 shape: torch.Size([2, 16, 64, 64, 32])
After enc2 shape: torch.Size([2, 32, 32, 32, 16])
After enc3 shape: torch.Size([2, 64, 16, 16, 8])
After enc4 shape: torch.Size([2, 128, 8, 8, 4])
enc4_flat shape: torch.Size([2, 32768])
latent_params shape: torch.Size([2, 16])
mu shape: torch.Size([2, 8])
log_var shape: torch.Size([2, 8])
z shape: torch.Size([2, 8])
enc4_decoded shape: torch.Size([2, 32768])
enc4 reshaped shape: torch.Size([2, 128, 8, 8, 4])
weibull_params shape: torch.Size([2, 2])
dec4 shape: torch.Size([2, 64, 16, 16, 8])
dec3 shape: torch.Size([2, 32, 32, 32, 16])
dec2 shape: torch.Size([2, 16, 64, 64, 32])
dec1 shape: torch.Size([2, 1, 128, 128, 64])


In [ ]:
# Print the output shapes
for key, value in output_data.items():
    print(f"{key} shape: {value.shape}")

latent_sample shape: torch.Size([2, 8])
weibull_params shape: torch.Size([2, 2])
mu shape: torch.Size([2, 8])
log_var shape: torch.Size([2, 8])
reconstruction shape: torch.Size([2, 1, 128, 128, 64])


In [ ]:
time_info = model.predict_time(input_tensor)

Input shape: torch.Size([2, 1, 128, 128, 64])
After enc1 shape: torch.Size([2, 16, 64, 64, 32])
After enc2 shape: torch.Size([2, 32, 32, 32, 16])
After enc3 shape: torch.Size([2, 64, 16, 16, 8])
After enc4 shape: torch.Size([2, 128, 8, 8, 4])
enc4_flat shape: torch.Size([2, 32768])
latent_params shape: torch.Size([2, 16])
mu shape: torch.Size([2, 8])
log_var shape: torch.Size([2, 8])
z shape: torch.Size([2, 8])
enc4_decoded shape: torch.Size([2, 32768])
enc4 reshaped shape: torch.Size([2, 128, 8, 8, 4])
weibull_params shape: torch.Size([2, 2])
dec4 shape: torch.Size([2, 64, 16, 16, 8])
dec3 shape: torch.Size([2, 32, 32, 32, 16])
dec2 shape: torch.Size([2, 16, 64, 64, 32])
dec1 shape: torch.Size([2, 1, 128, 128, 64])


In [ ]:
print("Latent Vector (z):", time_info['z'])
print("Weibull Parameters (time_params):", time_info['time_params'])
print("Sampled Time (time_samples):", time_info['time_samples'])


Latent Vector (z): [[ 0.08531892  0.34620687  2.062882   -1.6217979  -2.2246227   0.83534956
  -0.7993088  -1.9503777 ]
 [-0.8043294  -0.5259456   0.76776767 -1.356632   -0.49667412  1.3170807
  -0.15564978  3.1529782 ]]
Weibull Parameters (time_params): [[0.5902811  0.57451755]
 [0.6011775  0.662761  ]]
Sampled Time (time_samples): [0.94490194 0.6806333 ]


In [ ]:
just_time_info = model.predict_just_time(input_tensor)
print("Sampled Time (time_samples):", time_info['time_samples'])

Input shape: torch.Size([2, 1, 128, 128, 64])
After enc1 shape: torch.Size([2, 16, 64, 64, 32])
After enc2 shape: torch.Size([2, 32, 32, 32, 16])
After enc3 shape: torch.Size([2, 64, 16, 16, 8])
After enc4 shape: torch.Size([2, 128, 8, 8, 4])
enc4_flat shape: torch.Size([2, 32768])
latent_params shape: torch.Size([2, 16])
mu shape: torch.Size([2, 8])
log_var shape: torch.Size([2, 8])
z shape: torch.Size([2, 8])
enc4_decoded shape: torch.Size([2, 32768])
enc4 reshaped shape: torch.Size([2, 128, 8, 8, 4])
weibull_params shape: torch.Size([2, 2])
dec4 shape: torch.Size([2, 64, 16, 16, 8])
dec3 shape: torch.Size([2, 32, 32, 32, 16])
dec2 shape: torch.Size([2, 16, 64, 64, 32])
dec1 shape: torch.Size([2, 1, 128, 128, 64])
Sampled Time (time_samples): [0.94490194 0.6806333 ]


In [ ]:
import numpy as np
import torch

# Set a random seed for reproducibility
np.random.seed(42)
torch.manual_seed(42)

# Define batch size
batch_size = 3

# Generate dummy time_train data (e.g., survival times)
# Let's assume time points are between 0 and 100
time_train = np.random.uniform(0, 100, size=batch_size)

# Generate dummy x_val data
# Shape: (3, 1, 128, 128, 64)
x_val = torch.rand(batch_size, 1, 128, 128, 64)

# Generate dummy censor_val data
# Let's assume a 50% censoring rate
#censor_val = np.random.randint(0, 2, size=batch_size)
censor_val = np.array([1, 1, 0]) 
# Print the shapes of the generated data
print("time_train shape:", time_train.shape)  # Expect (3,)
print("x_val shape:", x_val.shape)            # Expect (3, 1, 128, 128, 64)
print("censor_val shape:", censor_val.shape)  # Expect (3,)


time_train shape: (3,)
x_val shape: torch.Size([3, 1, 128, 128, 64])
censor_val shape: (3,)


In [ ]:
time_train

array([37.45401188, 95.07143064, 73.19939418])

In [ ]:
censor_val

array([1, 1, 0])

In [ ]:
risk = model.calculate_risk(time_train, x_val, censor_val)

Input shape: torch.Size([3, 1, 128, 128, 64])
After enc1 shape: torch.Size([3, 16, 64, 64, 32])
After enc2 shape: torch.Size([3, 32, 32, 32, 16])
After enc3 shape: torch.Size([3, 64, 16, 16, 8])
After enc4 shape: torch.Size([3, 128, 8, 8, 4])
enc4_flat shape: torch.Size([3, 32768])
latent_params shape: torch.Size([3, 16])
mu shape: torch.Size([3, 8])
log_var shape: torch.Size([3, 8])
z shape: torch.Size([3, 8])
enc4_decoded shape: torch.Size([3, 32768])
enc4 reshaped shape: torch.Size([3, 128, 8, 8, 4])
weibull_params shape: torch.Size([3, 2])
dec4 shape: torch.Size([3, 64, 16, 16, 8])
dec3 shape: torch.Size([3, 32, 32, 32, 16])
dec2 shape: torch.Size([3, 16, 64, 64, 32])
dec1 shape: torch.Size([3, 1, 128, 128, 64])


is_monotonic is deprecated and will be removed in a future version. Use is_monotonic_increasing instead.


In [ ]:
risk

((0.3684, 1.0, 1.0), (0.1339, 0.6638592724074938, 0.9825))

In [ ]:
import scipy
print(scipy.__version__)


1.10.1


In [ ]:
from scipy.integrate import trapz

# Compute the integral using the trapezoidal rule
integral = trapz(y, x)
print(f"The integral using trapezoidal rule is approximately: {integral}")


NameError: name 'y' is not defined

In [ ]:
import scipy.integrate
print(dir(scipy.integrate))  # This will list all available functions and attributes


['AccuracyWarning', 'BDF', 'DOP853', 'DenseOutput', 'IntegrationWarning', 'LSODA', 'OdeSolution', 'OdeSolver', 'RK23', 'RK45', 'Radau', '__all__', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', '_bvp', '_dop', '_ivp', '_lsoda', '_ode', '_odepack', '_odepack_py', '_quad_vec', '_quadpack', '_quadpack_py', '_quadrature', '_vode', 'complex_ode', 'cumtrapz', 'cumulative_trapezoid', 'dblquad', 'dop', 'fixed_quad', 'lsoda', 'newton_cotes', 'nquad', 'ode', 'odeint', 'odepack', 'quad', 'quad_vec', 'quadpack', 'quadrature', 'romb', 'romberg', 'simps', 'simpson', 'solve_bvp', 'solve_ivp', 'test', 'tplquad', 'trapezoid', 'trapz', 'vode']


In [ ]:
from scipy.integrate import simps
import numpy as np
from scipy.integrate import simps

# Example data
x = np.linspace(0, 10, 100)
y = np.sin(x)

# Compute the integral using Simpson's rule
integral = simps(y, x)
print(f"The integral is approximately: {integral}")


ImportError: cannot import name 'simps' from 'scipy.integrate' (/home/nikhil/Downloads/tcia_download/g17p3/lib/python3.10/site-packages/scipy/integrate/__init__.py)

In [ ]:
import torch
import torch.nn as nn
import numpy as np


class UNet3D(nn.Module):
    def __init__(self, in_channels, out_channels, base_channels, latent_dim=None):
        super(UNet3D, self).__init__()
        
        self.latent_dim = latent_dim
        
        # Encoder and decoder layers
        self.enc1 = Encoder3D(in_channels, base_channels)
        self.enc2 = Encoder3D(base_channels, base_channels * 2)
        self.enc3 = Encoder3D(base_channels * 2, base_channels * 4)
        self.enc4 = Encoder3D(base_channels * 4, base_channels * 8)
        
        if latent_dim is not None:
            self.latent_space = LatentSpaceGaussian(latent_dim)
            self.latent_mu = nn.Linear(base_channels * 8 * 8 * 8 * 4, self.latent_space.latent_params)
            self.fc_decode = nn.Linear(latent_dim, base_channels * 8 * 8 * 8 * 4)
        
        self.dec4 = Decoder3D(base_channels * 8, base_channels * 4)
        self.dec3 = Decoder3D(base_channels * 4, base_channels * 2)
        self.dec2 = Decoder3D(base_channels * 2, base_channels)
        self.dec1 = Decoder3D(base_channels, out_channels)

        if latent_dim is not None:
            self.time_decoder = TimeDecoder(latent_dim)
    
    def forward(self, x):
        enc1 = self.enc1(x)
        enc2 = self.enc2(enc1)
        enc3 = self.enc3(enc2)
        enc4 = self.enc4(enc3)
        
        if self.latent_dim is not None:
            enc4_flat = enc4.view(enc4.size(0), -1)
            latent_params = self.latent_mu(enc4_flat)
            mu, log_var = self.latent_space.get_latent_params(latent_params)
            z = self.latent_space.sample_latent((mu, log_var))
            enc4 = self.fc_decode(z)
            enc4 = enc4.view(enc4.size(0), -1, 8, 8, 4)
            kl_divergence = self.latent_space.kl_loss((mu, log_var))
            weibull_params = self.time_decoder(z)
        
        dec4 = self.dec4(enc4)
        dec3 = self.dec3(dec4 + enc3)
        dec2 = self.dec2(dec3 + enc2)
        dec1 = self.dec1(dec2 + enc1)

        if self.latent_dim is not None:
            return dec1, weibull_params, kl_divergence
        else:
            return dec1

    def predict_time(self, x, just_cov=False, device=torch.device('cpu')):
        """Predicts Weibull time distribution parameters and samples for survival analysis."""
        if not just_cov:
            cov = x.drop(['time', 'event'], axis=1)
        else:
            cov = x.copy()

        # Forward pass to get latent representations and time predictions
        out_params = self.forward(cov.to(device))
        z = out_params[1]  # Assuming 'z' is returned as the second output
        time_params = self.time_decoder(z)

        # Generate samples from time distribution
        time_samples = sample_from_dist(time_params, [self.time_dist])

        return {
            'z': z.detach().cpu().numpy(),
            'time_params': time_params.cpu().numpy(),
            'time_samples': time_samples
        }

    def predict_just_time(self, x):
        """Predicts only the time samples based on the input covariates."""
        return self.predict_time(x, just_cov=True)['time_samples']


    def calculate_risk(self, time_train, x_val, censor_val):
        """Calculate risk CDF at specified time points, compute C-index and IBS with confidence intervals."""
        times = np.unique(time_train)
        out_data = self.predict_time(x_val)
        pred_risk = np.zeros([out_data['time_params'].shape[0], len(times)])

        for sample in range(pred_risk.shape[0]):
            if self.time_dist[0] == 'weibull':
                alpha = out_data['time_params'][sample, 0]
                lam = out_data['time_params'][sample, 1]
                pred_risk[sample, :] = 1 - np.exp(-np.power(times / lam, alpha))
            else:
                raise RuntimeError('Unknown time distribution for risk calculation.')

        # Calculate C-index and IBS along with confidence intervals
        ci, ibs = obtain_c_index(pd.DataFrame(1 - pred_risk.T, index=times), np.array(x_val['time']), censor_val)
        ci_conf_intervals = bern_conf_interval(len(np.array(x_val['time'])), ci)
        ibs_conf_intervals = bern_conf_interval(len(np.array(x_val['time'])), ibs, ibs=True)

        return ci_conf_intervals, ibs_conf_intervals
     

    def fit_epoch(self, data, optimizer, batch_size=64, device=torch.device('cpu')):
        epoch_results = {'loss_tr': 0.0, 'loss_va': 0.0, 'kl_tr': 0.0, 'kl_va': 0.0, 'll_cov_tr': 0.0, 'll_cov_va': 0.0,
                         'll_time_tr': 0.0, 'll_time_va': 0.0, 'ci_va': 0.0, 'ibs_va': 0.0}

        for batch in range(n_batches):
    


    def fit(self, data, train_params):
        training_stats = {'loss_tr': [], 'loss_va': [], 'kl_tr': [], 'kl_va': [], 'll_cov_tr': [], 'll_cov_va': [],
                          'll_time_tr': [], 'll_time_va': [], 'ci_va': [], 'ibs_va': []}

        optimizer = torch.optim.Adam(self.parameters(), lr=train_params['lr'])

        for epoch in range(train_params['n_epochs']):       


IndentationError: expected an indented block after 'for' statement on line 109 (4192251332.py, line 113)

In [ ]:
import torch

# Generate a random tensor
random_input = torch.tensor([100.0])
  # Scaling by 10 to get a wider range of values
softplus_output = torch.nn.functional.softplus(random_input)

random_input.item(), softplus_output.item()